In [5]:
# =============================
# Imports
# =============================
import os
import json
import yfinance as yf
import pandas as pd
from datetime import datetime, timedelta

# =============================
# Configuration
# =============================
OUTPUT_FOLDER = "data"
RAW_COMBINED_CSV = os.path.join(OUTPUT_FOLDER, "etl-data-raw.csv")

os.makedirs(OUTPUT_FOLDER, exist_ok=True)

# List of tickers to fetch
etf_list = ["AAPL","ALLY","AMZN","ARKK","FBCG","FMAG","GGLL",
            "GOOGL","HSBC","MGK","MSFT","MSFU","NVDA","OEF",
            "QQQ","QQQJ","QQUP","QQXL","QTOP","SPLG","TOPT",
            "TQQQ","UPRO","VGT","VTI","XLG"]


In [6]:
#def safe_str_date(dt):
#    if pd.isna(dt):
#        return ""
#    if hasattr(dt, "strftime"):
#        return dt.strftime("%Y-%m-%d")
#    return str(dt)

In [7]:
rows = []
for sym in etf_list:
    print(f"[fetch] {sym}")
    try:
        t = yf.Ticker(sym)
        df = t.history(period="20y", interval="1d", auto_adjust=True)
        if df is None or df.empty:
            print(f"  no data for {sym}, skipping")
            continue

        df = df.reset_index()
        df["Symbol"] = sym

        keep_cols = ["Symbol", "Date", "Open", "High", "Low", "Close", "Volume"]
        for c in keep_cols:
            if c not in df.columns:
                df[c] = pd.NA
        df = df[keep_cols]

        rows.append(df)
    except Exception as e:
        print(f"  error fetching {sym}: {e}")

if rows:
    combined = pd.concat(rows, ignore_index=True)
    combined.to_csv(RAW_COMBINED_CSV, index=False)
    print(f"Wrote raw combined CSV -> {RAW_COMBINED_CSV}")
else:
    print("No data downloaded.")


[fetch] AAPL
[fetch] ALLY
[fetch] AMZN
[fetch] ALLY
[fetch] AMZN
[fetch] ARKK
[fetch] FBCG
[fetch] ARKK
[fetch] FBCG
[fetch] FMAG
[fetch] GGLL
[fetch] FMAG
[fetch] GGLL
[fetch] GOOGL
[fetch] GOOGL
[fetch] HSBC
[fetch] HSBC
[fetch] MGK
[fetch] MSFT
[fetch] MGK
[fetch] MSFT
[fetch] MSFU
[fetch] NVDA
[fetch] MSFU
[fetch] NVDA
[fetch] OEF
[fetch] OEF
[fetch] QQQ
[fetch] QQQ
[fetch] QQQJ
[fetch] QQUP
[fetch] QQXL
[fetch] QQQJ
[fetch] QQUP
[fetch] QQXL
[fetch] QTOP
[fetch] SPLG
[fetch] QTOP
[fetch] SPLG
[fetch] TOPT
[fetch] TQQQ
[fetch] TOPT
[fetch] TQQQ
[fetch] UPRO
[fetch] VGT
[fetch] UPRO
[fetch] VGT
[fetch] VTI
[fetch] VTI
[fetch] XLG
[fetch] XLG
Wrote raw combined CSV -> data\etl-data-raw.csv
Wrote raw combined CSV -> data\etl-data-raw.csv


In [8]:
# Export per-ticker CSV files for UI consumption
import os
import json

# Read the BOD-enhanced CSV we just wrote
bod_path = os.path.join(OUTPUT_FOLDER, "etl-data-bod.csv")
combined = pd.read_csv(bod_path)

# Ensure avg_daily_price exists (compute if missing)
if 'avg_daily_price' not in combined.columns:
    combined[["Open", "High", "Low", "Close"]] = combined[["Open", "High", "Low", "Close"]].apply(pd.to_numeric, errors='coerce')
    combined['avg_daily_price'] = combined[["Open", "High", "Low", "Close"]].mean(axis=1).round(4)

# Create output directory for tickers
tickers_dir = os.path.join(OUTPUT_FOLDER, 'tickers')
os.makedirs(tickers_dir, exist_ok=True)

# Write one CSV per Symbol and build an index
index = {}
for sym, grp in combined.groupby('Symbol'):
    fname = f"{sym}.csv"
    path = os.path.join(tickers_dir, fname)

    # Reorder columns: move Shares_Purchased and Total_Value_Purchased to the left of any BOD* columns
    cols = list(grp.columns)

    # Helper to find the actual column name when variants exist (underscores vs spaces, different casing)
    def find_variant(preferred_names, available_cols):
        for n in preferred_names:
            if n in available_cols:
                return n
        # try normalized match
        norm_map = {c.lower().replace(' ', '').replace('_', ''): c for c in available_cols}
        for n in preferred_names:
            key = n.lower().replace(' ', '').replace('_', '')
            if key in norm_map:
                return norm_map[key]
        return None

    shares_col = find_variant(['Shares_Purchased', 'Shares Purchased', 'shares_purchased', 'SharesPurchased'], cols)
    total_col = find_variant(['Total_Value_Purchased', 'Total Value Purchased', 'TotalValuePurchased', 'total_value_purchased'], cols)

    # Find BOD-like columns (start with 'BOD' or contain 'BOD[' )
    bod_cols = [c for c in cols if isinstance(c, str) and (c.startswith('BOD') or 'BOD[' in c or c.upper().startswith('BOD'))]

    if (shares_col or total_col):
        # Build new column order
        remaining = [c for c in cols if c not in (shares_col, total_col)]
        if bod_cols:
            # find first index of any bod col in remaining
            bod_indices = [remaining.index(b) for b in bod_cols if b in remaining]
            insert_idx = min(bod_indices) if bod_indices else 0
        else:
            # No BOD columns found: place at front
            insert_idx = 0

        insert_cols = []
        if shares_col:
            insert_cols.append(shares_col)
        if total_col and total_col != shares_col:
            insert_cols.append(total_col)

        new_order = remaining[:insert_idx] + insert_cols + remaining[insert_idx:]
        # Reindex grp to new order (only include columns that actually exist)
        new_order = [c for c in new_order if c in grp.columns]
        grp = grp.reindex(columns=new_order)

    # Write CSV and record index path
    grp.to_csv(path, index=False)
    # Use a web-friendly, repo-relative path so a hosted UI can fetch it
    index[sym] = f"{OUTPUT_FOLDER}/tickers/{fname}"

# Save index.json in data/
index_path = os.path.join(OUTPUT_FOLDER, 'tickers_index.json')
with open(index_path, 'w') as f:
    json.dump(index, f, indent=2)

print(f"Wrote {len(index)} ticker files -> {tickers_dir}")
print(f"Index file -> {index_path}")

# Show sample head for the first ticker
sample_sym = next(iter(index))
sample_path = os.path.join(tickers_dir, f"{sample_sym}.csv")
print(f"\nSample head for {sample_sym}:")
print(pd.read_csv(sample_path).head().to_string())


FileNotFoundError: [Errno 2] No such file or directory: 'data\\etl-data-bod.csv'